## Fine-tuning

In [ ]:
%pip install transformers[torch] datasets

In [ ]:
import transformers
import torch

from transformers import AutoModel, AutoTokenizer
from datasets import load_dataset
from transformers import Trainer, TrainingArguments
from transformers import AutoTokenizer
from transformers import AutoModelForCausalLM

In [ ]:
model = "openai-community/gpt2"
block_size = 2**5 # this is small for demonstration purposes, you can increase it

In [ ]:
transformers.enable_full_determinism( 0 )

In [ ]:
datasets = load_dataset("text", data_files={"train": './data/*.txt', "validation": './data/*.txt'})

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(model, use_fast=True)
tokenizer.pad_token = tokenizer.eos_token

def tokenize_function(examples):
            return tokenizer(examples["text"])

tokenized_datasets = datasets.map(tokenize_function, batched=True, num_proc=4, remove_columns=["text"])


In [ ]:
def group_texts(examples):
    concatenated_examples = {k: sum(examples[k], []) for k in examples.keys()}
    total_length = len(concatenated_examples[list(examples.keys())[0]])
    total_length = (total_length // block_size) * block_size
    result = {
        k: [t[i : i + block_size] for i in range(0, total_length, block_size)]
        for k, t in concatenated_examples.items()
    }
    result["labels"] = result["input_ids"].copy()
    return result

lm_datasets = tokenized_datasets.map(
    group_texts,
    batched=True,
    batch_size=1000,
    num_proc=4
)

In [ ]:
model = AutoModelForCausalLM.from_pretrained(model)

In [ ]:
training_args = TrainingArguments(
    eval_strategy="epoch",
    learning_rate=2e-5,
    num_train_epochs=2, ## this is small for demonstration purposes, you can increase it
    weight_decay=0.01,
    output_dir= f"./models/finetuned",
    logging_dir= f"./models/finetuned-logs",
    logging_steps=10,
)

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=lm_datasets["train"],
    eval_dataset=lm_datasets["validation"],
)

trainer.train()

In [ ]:
trainer.save_model(f"./models/finetuned")
tokenizer.save_pretrained(f"./models/finetuned")